In [4]:
#!pip install librosa tqdm scikit-learn

In [1]:
MY_CONFIG = {
    "batch_size": 512, # Crank up batch size to match official baseline specs
    "epochs": 50, # Keep training cycles short for the initial run
    "input_dim": 320,
}

BASE_PATH = "/content/mimii_data/fan"
SELECTED_IDS = ["id_00", "id_02", "id_04", "id_06"]
ZIP_PATH = "/content/drive/MyDrive/0_dB_fan.zip"
LOCAL_DATA_PATH = "/content/mimii_data"
SAVE_DIR = "/content/drive/MyDrive/MIMII_Baseline_Results"

In [20]:
import os
import sys
import glob
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from tqdm import tqdm
from sklearn import metrics
import logging

# ==========================================
# 1. Feature Extraction (Mirrors official Baseline.py)
# ==========================================
def file_to_vector_array(file_name, n_mels=64, frames=5, n_fft=1024, hop_length=512, power=2.0):
    # Load audio; sr=None keeps the native sampling rate
    y, sr = librosa.load(file_name, sr=None, mono=True)

    # Calculate Mel Spectrogram
    mel_spectrogram = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=power
    )

    # Convert to Log-Mel using the specific scaling from the official repo
    log_mel_spectrogram = 20.0 / power * np.log10(mel_spectrogram + sys.float_info.epsilon)

    # Slide a window across frames and concatenate them into a single long vector
    vectorarray_size = log_mel_spectrogram.shape[1] - frames + 1
    if vectorarray_size < 1:
        return np.empty((0, n_mels * frames), float)

    dims = n_mels * frames
    vectorarray = np.zeros((vectorarray_size, dims), float)
    for t in range(frames):
        vectorarray[:, n_mels * t: n_mels * (t + 1)] = log_mel_spectrogram[:, t: t + vectorarray_size].T

    return vectorarray

# ==========================================
# 2. The Autoencoder Model (64-64-8-64-64 structure)
# ==========================================
class MIMII_Baseline_AE(nn.Module):
    def __init__(self, input_dim):
        super(MIMII_Baseline_AE, self).__init__()
        # Encoder: Compresses the 320D input down to an 8D bottleneck
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 8),
            nn.ReLU()
        )
        # Decoder: Reconstructs the original input dimensions from the bottleneck
        self.decoder = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim) # No activation on the final layer per baseline specs
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

# ==========================================
# 3. Training & Evaluation Pipeline
# ==========================================
def train_baseline(target_dir, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Data Prep ---
    normal_files = sorted(glob.glob(f"{target_dir}/normal/*.wav"))
    abnormal_files = sorted(glob.glob(f"{target_dir}/abnormal/*.wav"))

    # Split: train on one set of normal files, test on others + anomalies
    train_files = normal_files[len(abnormal_files):]
    eval_files = normal_files[:len(abnormal_files)] + abnormal_files
    eval_labels = [0] * len(normal_files[:len(abnormal_files)]) + [1] * len(abnormal_files)

    print(f"Crunching features for {target_dir}...")
    all_features = []
    for f in tqdm(train_files):
        vec = file_to_vector_array(f)
        all_features.append(vec)

    train_x = np.concatenate(all_features, axis=0)
    train_x = torch.FloatTensor(train_x)

    # Mimicking Keras' 10% validation split
    dataset = TensorDataset(train_x, train_x) 
    val_size = int(len(dataset) * 0.1)
    train_size = len(dataset) - val_size
    train_db, val_db = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_db, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_db, batch_size=config['batch_size'], shuffle=False)

    # --- Setup Model ---
    input_dim = config['input_dim']
    model = MIMII_Baseline_AE(input_dim).to(device)
    optimizer = optim.Adam(model.parameters())
    criterion = nn.MSELoss()

    # --- Main Training Loop ---
    print(f"Starting training...")
    for epoch in range(config['epochs']):
        model.train()
        t_loss = 0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, x)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1} | Loss: {t_loss/len(train_loader):.6f}")

    # --- Evaluation (ROC-AUC calculation) ---
    print(f"Running evaluation...")
    model.eval()
    y_pred = []
    with torch.no_grad():
        for file_name in tqdm(eval_files):
            data = file_to_vector_array(file_name)
            data_tensor = torch.FloatTensor(data).to(device)

            # Reconstruct and find the error
            reconstructed = model(data_tensor)
            # Calculate MSE for each frame, then average across the whole file
            mse_per_frame = torch.mean((data_tensor - reconstructed)**2, dim=1)
            file_error = torch.mean(mse_per_frame).cpu().item()
            y_pred.append(file_error)

    auc = metrics.roc_auc_score(eval_labels, y_pred)

    # Save logic
    model_name = f"baseline_fan_{os.path.basename(target_dir)}.pth"
    full_save_path = os.path.join("/content/drive/MyDrive/MIMII_Baseline_Results", model_name)

    torch.save({
        'epoch': config['epochs'],
        'model_state_dict': model.state_dict(),
        'auc': auc,
        'input_dim': 320 
    }, full_save_path)

    print(f"💾 Checkpoint saved to: {full_save_path}")
    return auc

In [2]:
import os
from google.colab import drive

# Hooking up Google Drive so we can access the stored zip files
drive.mount('/content/drive')

# Let's make sure we actually need to unzip the data before starting
if not os.path.exists(LOCAL_DATA_PATH):
    os.makedirs(LOCAL_DATA_PATH, exist_ok=True)
    print(f"📦 Extracting dataset to the local runtime: {LOCAL_DATA_PATH}...")
    
    # Use 'quiet' mode (-q) so we don't drown the console in thousands of filenames
    !unzip -q {ZIP_PATH} -d {LOCAL_DATA_PATH}
    print("✅ Extraction finished!")
else:
    # If the folder is already there, we skip the unzip to save time
    print("ℹ️ Dataset already exists locally. Moving on.")

ModuleNotFoundError: No module named 'google.colab'

In [15]:
# Check if our results folder is already there
if not os.path.exists(SAVE_DIR):
    try:
        # Try to build the directory path; exist_ok=True is a safety net
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"✅ Output directory is ready: {SAVE_DIR}")
    except Exception as e:
        # If this fails, it's usually because the Drive isn't mounted or permissions are wonky
        print(f"❌ Couldn't create the folder. Is your Drive actually mounted? Error: {e}")

✅ 已成功创建目录: /content/drive/MyDrive/MIMII_Baseline_Results


In [16]:
# ==========================================
# Batch runner for the four Machine IDs
# ==========================================

# Make sure the train_baseline function and AE class are already loaded in memory
baseline_results = {}

print("🚀 Starting the official Baseline evaluation...")
print("-" * 50)

for mid in SELECTED_IDS:
    target_path = os.path.join(BASE_PATH, mid)

    # Path fix: if the standard path is broken, try to find where the ID actually lives
    if not os.path.exists(target_path):
        print(f"⚠️ Path {target_path} not found. Searching for it...")
        # Zip extractions often nest folders unexpectedly; this handles that.
        search_path = glob.glob(f"{LOCAL_DATA_PATH}/**/{mid}", recursive=True)
        if search_path:
            target_path = search_path[0]
        else:
            print(f"❌ Could not find data for {mid}. Check your zip extraction structure.")
            continue

    print(f"\n[Processing] Machine ID: {mid}")
    try:
        # Run the training and eval (using the 50-epoch config we set up earlier)
        auc_score = train_baseline(target_path, MY_CONFIG)
        baseline_results[mid] = auc_score
        print(f"✨ {mid} Result: AUC = {auc_score:.4f}")
    except Exception as e:
        print(f"💥 {mid} failed: {str(e)}")

# ==========================================
# Final Performance Report
# ==========================================
print("\n" + "="*50)
print("🏆 MIMII OFFICIAL BASELINE EXPERIMENT REPORT")
print("-" * 50)
print(f"{'Machine ID':<15} | {'Baseline AUC (0dB)':<15}")
print("-" * 50)
for mid in SELECTED_IDS:
    score = baseline_results.get(mid, "N/A")
    score_str = f"{score:.4f}" if isinstance(score, float) else score
    print(f"{mid:<15} | {score_str:<15}")
print("="*50)

🚀 开始官方 Baseline 评估流程...
--------------------------------------------------

[处理中] Machine ID: id_00
Loading features for /content/mimii_data/fan/id_00...


100%|██████████| 604/604 [00:26<00:00, 22.56it/s]


Training Baseline...
Epoch 10 | Loss: 11.060430
Epoch 20 | Loss: 11.056760
Epoch 30 | Loss: 10.939166
Epoch 40 | Loss: 10.409382
Epoch 50 | Loss: 10.355296
Evaluating...


100%|██████████| 814/814 [00:26<00:00, 31.22it/s]


💾 权重已保存至: /content/drive/MyDrive/MIMII_Baseline_Results/baseline_fan_id_00.pth
✨ id_00 结果: AUC = 0.6071

[处理中] Machine ID: id_02
Loading features for /content/mimii_data/fan/id_02...


...

💾 权重已保存至: /content/drive/MyDrive/MIMII_Baseline_Results/baseline_fan_id_04.pth
✨ id_04 结果: AUC = 0.7396

[处理中] Machine ID: id_06
Loading features for /content/mimii_data/fan/id_06...


100%|██████████| 654/654 [00:21<00:00, 31.14it/s]


Training Baseline...
Epoch 10 | Loss: 8.383632
Epoch 20 | Loss: 6.410928
Epoch 30 | Loss: 6.233908
Epoch 40 | Loss: 5.957648
Epoch 50 | Loss: 5.726080
Evaluating...


100%|██████████| 722/722 [00:24<00:00, 28.97it/s]

💾 权重已保存至: /content/drive/MyDrive/MIMII_Baseline_Results/baseline_fan_id_06.pth
✨ id_06 结果: AUC = 0.9791

🏆 MIMII OFFICIAL BASELINE EXPERIMENT REPORT
--------------------------------------------------
Machine ID      | Baseline AUC (0dB)
--------------------------------------------------
id_00           | 0.6071         
id_02           | 0.8244         
id_04           | 0.7396         
id_06           | 0.9791         


In [22]:
# Function to bring a trained model back to life from a saved file
def load_trained_model(model_path, device):
    # Grab the input dimensions from our config to build the architecture
    input_dim = MY_CONFIG['input_dim']
    model = MIMII_Baseline_AE(input_dim=input_dim).to(device)
    
    # Load the checkpoint file; we use map_location so it works on both CPU and GPU
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    
    # Check if the weights are tucked inside a dictionary or saved directly
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    
    # Set the model to 'eval' mode so it's ready for inference (no training updates)
    model.eval()
    return model

In [24]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# ==========================================
# 1. Scoring Logic: Calculating MSE for every file
# ==========================================
def get_eval_scores(model, eval_files, device):
    model.eval()
    y_pred_scores = []
    print(f"Calculating reconstruction error for {len(eval_files)} samples...")

    with torch.no_grad():
        for file_name in tqdm(eval_files):
            # Extract features exactly like we did during training: (frames, 320)
            data = file_to_vector_array(file_name)
            data_tensor = torch.FloatTensor(data).to(device)

            # Let the model try to rebuild the audio features
            reconstructed = model(data_tensor)

            # Official approach: Calculate MSE per frame, then average across the whole file
            # Formula: mean((input - output)^2)
            mse_per_frame = torch.mean((data_tensor - reconstructed)**2, dim=1)
            file_error = torch.mean(mse_per_frame).cpu().item()
            y_pred_scores.append(file_error)

    return np.array(y_pred_scores)

# ==========================================
# 2. Finding the Sweet Spot: Optimizing the F1-Score Threshold
# ==========================================
def find_best_threshold(y_true, y_scores):
    # Search through 1000 candidate thresholds between the min and max scores
    thresholds = np.linspace(min(y_scores), max(y_scores), 1000)
    best_f1 = 0
    best_threshold = 0
    best_metrics = {}

    for t in thresholds:
        # If the MSE score is above our threshold 't', we flag it as an anomaly (1)
        y_pred_binary = [1 if s > t else 0 for s in y_scores]

        # Calculate standard classification metrics
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred_binary, average='binary', zero_division=0
        )

        # Keep track of the threshold that gives us the best balance (F1)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t
            best_metrics = {
                "threshold": t,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "accuracy": accuracy_score(y_true, y_pred_binary),
                "conf_matrix": confusion_matrix(y_true, y_pred_binary)
            }

    return best_metrics

# ==========================================
# 3. Main Runner: Classification for all Machine IDs
# ==========================================
def run_classification_pipeline(target_id):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = f"/content/drive/MyDrive/MIMII_Baseline_Results/baseline_fan_{target_id}.pth"

    print(f"\n--- Running Classification for {target_id} ---")
    
    # Step 1: Load the checkpoint
    model = load_trained_model(model_path, device)

    # Step 2: Set up the test set
    target_dir = os.path.join(BASE_PATH, target_id)
    normal_files = sorted(glob.glob(f"{target_dir}/normal/*.wav"))
    abnormal_files = sorted(glob.glob(f"{target_dir}/abnormal/*.wav"))

    # Balance the test set: Same number of normal and abnormal samples
    eval_files = normal_files[:len(abnormal_files)] + abnormal_files
    eval_labels = [0] * len(normal_files[:len(abnormal_files)]) + [1] * len(abnormal_files)

    # Step 3: Compute the "Anomaly Scores" (MSE)
    scores = get_eval_scores(model, eval_files, device)

    # Step 4: Find the threshold that yields the best classification performance
    report = find_best_threshold(eval_labels, scores)

    # Step 5: Print the final results
    print("\n" + "="*45)
    print(f"📊 Final Report for {target_id} (Best F1 Threshold)")
    print("-" * 45)
    print(f"Best Threshold (MSE): {report['threshold']:.6f}")
    print(f"Accuracy:            {report['accuracy']:.4f}")
    print(f"Precision:           {report['precision']:.4f}")
    print(f"Recall:              {report['recall']:.4f}")
    print(f"F1-Score:            {report['f1_score']:.4f}")
    print("-" * 45)
    print(f"Confusion Matrix:\n{report['conf_matrix']}")
    print("Key: [Top-Left: TN, Top-Right: FP, Bot-Left: FN, Bot-Right: TP]")
    print("="*45)

# Execute the pipeline for each selected ID
for id in SELECTED_IDS:
    run_classification_pipeline(id)

--- 正在处理 id_00 ---
正在计算 814 个评估样本的重构误差...


100%|██████████| 814/814 [00:25<00:00, 31.49it/s]



📊 id_00 最终分类报告 (最优 F1 阈值下)
---------------------------------------------
最优阈值 (MSE): 6.946039
准确率 (Accuracy): 0.5848
精确率 (Precision): 0.5502
召回率 (Recall):    0.9287
F1-Score:         0.6910
---------------------------------------------
混淆矩阵:
[[ 98 309]
 [ 29 378]]
注: [左上: TN, 右上: FP, 左下: FN, 右下: TP]
--- 正在处理 id_02 ---
正在计算 718 个评估样本的重构误差...


100%|██████████| 718/718 [00:20<00:00, 35.88it/s]



📊 id_02 最终分类报告 (最优 F1 阈值下)
---------------------------------------------
最优阈值 (MSE): 6.016226
准确率 (Accuracy): 0.6950
精确率 (Precision): 0.6411
召回率 (Recall):    0.8858
F1-Score:         0.7439
---------------------------------------------
混淆矩阵:
[[181 178]
 [ 41 318]]
注: [左上: TN, 右上: FP, 左下: FN, 右下: TP]
--- 正在处理 id_04 ---
正在计算 696 个评估样本的重构误差...


100%|██████████| 696/696 [00:17<00:00, 39.24it/s]



📊 id_04 最终分类报告 (最优 F1 阈值下)
---------------------------------------------
最优阈值 (MSE): 5.303186
准确率 (Accuracy): 0.6336
精确率 (Precision): 0.5774
召回率 (Recall):    0.9971
F1-Score:         0.7313
---------------------------------------------
混淆矩阵:
[[ 94 254]
 [  1 347]]
注: [左上: TN, 右上: FP, 左下: FN, 右下: TP]
--- 正在处理 id_06 ---
正在计算 722 个评估样本的重构误差...


100%|██████████| 722/722 [00:18<00:00, 38.76it/s]



📊 id_06 最终分类报告 (最优 F1 阈值下)
---------------------------------------------
最优阈值 (MSE): 7.010286
准确率 (Accuracy): 0.9058
精确率 (Precision): 0.8709
召回率 (Recall):    0.9529
F1-Score:         0.9101
---------------------------------------------
混淆矩阵:
[[310  51]
 [ 17 344]]
注: [左上: TN, 右上: FP, 左下: FN, 右下: TP]
